# Tiền xử lý dữ liệu cho baseline LSTM

**Mục tiêu:** LSTM là baseline **không có đồ thị** để đo đóng góp của GAT trong GAT-GRU. Để so sánh công bằng:

| Giữ giống hệt GAT-GRU | Khác GAT-GRU |
|---|---|
| Run, kịch bản, split (`modeling-common/splits.json`) | Không có cạnh, không có feature cạnh |
| 15 feature nút ứng viên, cách tính, warm-up, chuẩn hóa | Không có message passing giữa service |
| Target (`rps_in`, `cpu_cores`), horizon h = 6 | |
| Quy trình feature selection + Optuna (cùng ngân sách) | |

Notebook dùng chung module `../modeling-common/forecast_data.py` với GAT-GRU.

**Nguồn dữ liệu:**
- **Mặc định:** các run trong `../load-testing/results/` (đã có đủ dữ liệu LSTM cần). Hai mô hình vì vậy được **đánh giá trên cùng các run test**.
- **Tùy chọn** `INCLUDE_OWN_RUNS = True`: thêm các run do `lstm-load-testing/run-scenario.sh` tự thu (`results/`), **chỉ vào tập train**. Lưu ý: LSTM khi đó có nhiều dữ liệu train hơn GAT-GRU. Hãy ghi rõ điều này nếu dùng trong báo cáo.

**Quy trình:** `preprocess_lstm` → `feature_selection_lstm` → `train_lstm` → `compare_lstm_gatgru`.

**Chọn bộ dataset** bằng biến `DATASET` (đặt **giống nhau** ở mọi notebook của một lần thí nghiệm):

| `DATASET` | Nguồn | Kịch bản | Split | Thư mục đầu ra |
|---|---|---|---|---|
| `'load-testing'` | `load-testing/results/` (Locust, mô hình đóng) | normal, spike, bursty | `modeling-common/splits.json` | `processed/` |
| `'math'` | `math-load-testing/results/` (mô hình toán, luồng đến mở) | nhpp, mmpp, pareto, onoff | `modeling-common/splits_math.json` | `processed_math/` |

Hai bộ có split và thư mục kết quả riêng nên không ảnh hưởng nhau.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COMMON = (Path('..') / 'modeling-common').resolve()   # code dùng chung cho GAT-GRU và LSTM
sys.path.insert(0, str(COMMON))
import forecast_data as fd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
DATASET = 'load-testing'          # 'load-testing' | 'math'
DATASETS = {
    'load-testing': {'label': 'load-testing', 'results': Path('../load-testing/results'), 'tag': '',
                      'scenarios': ('normal', 'spike', 'bursty'), 'test_scenarios': ['spike']},
    'math': {'label': 'math-load-testing', 'results': Path('../math-load-testing/results'), 'tag': '_math',
              'scenarios': ('nhpp', 'mmpp', 'pareto', 'onoff'), 'test_scenarios': ['mmpp']},
}
DS = DATASETS[DATASET]
RESULTS = {DS['label']: DS['results']}
INCLUDE_AUTOSCALERS = None        # None = mọi run; ví dụ ['none'] hoặc ['none', 'hpa']
CORRECT_COVERAGE = False          # True: nhân rps_in/call_rate với 1/jaeger_coverage của run
JAEGER_COVERAGE_MIN = 0.9
REBUILD_SPLITS = False            # chỉ đặt True TRƯỚC khi bắt đầu tune/đánh giá
SPLIT_KW = dict(mode='by_run', frac={'train': 0.70, 'val': 0.15, 'test': 0.15},
                test_scenarios=DS['test_scenarios'], seed=42)
DEFAULT_WINDOW = 12               # chỉ cho dataset mặc định; window thật được tune ở train notebook

# Chưa có run thật -> dữ liệu demo dùng chung (modeling-common/demo_results*), để chạy thử pipeline.
DEMO = not any(DS['results'].glob('*/run_*/meta.json'))
if DEMO:
    demo_root = COMMON / f"demo_results{DS['tag']}"
    if not demo_root.exists():
        fd.make_demo_results(demo_root, scenarios=DS['scenarios'])
    RESULTS = {DS['label']: demo_root}
SPLITS_PATH = COMMON / f"splits{DS['tag']}{'_demo' if DEMO else ''}.json"
OUT = Path(f"processed{DS['tag']}{'_demo' if DEMO else ''}")   # đầu vào cho feature_selection_lstm / train_lstm
OUT.mkdir(exist_ok=True)
print('DATASET =', DATASET, '| DEMO =', DEMO, '| RESULTS =', RESULTS, '| OUT =', OUT.resolve())

INCLUDE_OWN_RUNS = False          # chỉ áp dụng cho DATASET = 'load-testing'
OWN_RESULTS = {'lstm': Path('results')}

## 1. Nạp dữ liệu và kiểm tra chất lượng

In [ ]:
runs = fd.load_runs(RESULTS)
assert runs, f'Không tìm thấy run nào trong {RESULTS}'
quality = fd.prepare_runs(runs)
if INCLUDE_AUTOSCALERS:
    runs = [r for r in runs if r['meta'].get('autoscaler', 'none') in INCLUDE_AUTOSCALERS]
bad = quality[(quality.prometheus_coverage < 0.9) | (quality.jaeger_coverage < JAEGER_COVERAGE_MIN)]
if len(bad):
    print('⚠️ Run có độ phủ thấp (cân nhắc loại hoặc chạy lại):')
    print(bad.to_string())
quality

In [ ]:
own = []
if INCLUDE_OWN_RUNS and not DEMO and DATASET == 'load-testing':
    own = fd.load_runs(OWN_RESULTS)
    if own:
        print('Run LSTM tự thu (chỉ dùng cho train):')
        print(fd.prepare_runs(own).to_string())

## Danh sách feature ứng viên và lý do

Mọi feature phải **quan sát được khi triển khai thật** (controller đọc từ Prometheus/Jaeger). Số user của Locust **không** là feature, chỉ dùng để kiểm tra độ phủ. Đây là tập **ứng viên**; bước *feature selection* sẽ loại các feature không cần thiết.

| Nhóm | Feature | Nguồn / công thức | Mục tiêu |
|---|---|---|---|
| load | `rps_in` | Jaeger: số span `server` / 10s | Nhu cầu tải trực tiếp; **target** chính, không bị chặn bởi limit |
| load | `rps_in_delta` | $rps_t - rps_{t-1}$ | Xu hướng tức thời, giúp nhận ra đầu spike/burst sớm |
| load | `rps_per_replica` | `rps_in / replicas` | Tải trên mỗi pod, quyết định mức bão hòa |
| perf | `latency_p50_ms`, `latency_p95_ms` | phân vị thời lượng span server | Tín hiệu SLO; độ trễ lan ngược theo chuỗi gọi |
| perf | `error_rate` | lỗi / request (HTTP 5xx, gRPC ≠ 0) | Quá tải / lỗi dây chuyền |
| resource | `cpu_cores` | cAdvisor, chỉ container ứng dụng | Nhu cầu tài nguyên; **target** thứ hai |
| resource | `cpu_util_request` | `cpu_cores / cpu_request_cores` | Đúng đại lượng HPA dùng |
| resource | `cpu_throttle_ratio` | throttled / CFS periods | Bão hòa ở limit: CPU đứng yên nhưng latency tăng |
| resource | `cpu_sidecar_cores` | CPU `istio-proxy` | Tỉ lệ với lưu lượng qua Envoy; tín hiệu tải dự phòng |
| resource | `mem_mib` | working set | Áp lực bộ nhớ, cảnh báo OOM |
| resource | `net_rx_kBps`, `net_tx_kBps` | cAdvisor network | Traffic, gồm cả TCP không có span (redis) |
| capacity | `replicas` | KSM replicas available | Năng lực hiện tại, biến PPO điều khiển |
| capacity | `restarts_delta` | Δ restarts | Sự kiện bất ổn |

Feature **bắt buộc giữ** (`fd.FORCED_FEATURES`): `rps_in`, `cpu_cores` (target được dùng làm đầu vào tự hồi quy) và `replicas` (biến PPO điều khiển).

## 2. Tạo mảng feature (không có cạnh)
`edges = []` nên `E` có M = 0 cạnh. Các feature nút được tính bằng đúng hàm của GAT-GRU.

In [ ]:
for r in runs + own:
    r['X'], r['E'], r['G'], r['tidy'] = fd.build_run_arrays(r, [], CORRECT_COVERAGE)
tidy_all = pd.concat([r['tidy'] for r in runs + own], ignore_index=True)
print('X', runs[0]['X'].shape)
tidy_all.groupby('service')[['rps_in', 'cpu_cores', 'latency_p95_ms', 'replicas']].mean().round(3)

## 3. Chia tập dùng chung
Split được tạo **chỉ từ các run dùng chung** (giống hệt GAT-GRU). Run LSTM tự thu (nếu có) được gán `train` trong bộ nhớ, không ghi vào file split dùng chung.

In [ ]:
splits = fd.load_or_create_splits(SPLITS_PATH, runs, rebuild=REBUILD_SPLITS, **SPLIT_KW)
splits = {**splits, 'runs': dict(splits['runs'])}
for r in own:
    splits['runs'][r['key']] = 'train'
runs = [r for r in runs + own if r['key'] in splits['runs']]
pd.DataFrame([{'scenario': r['scenario'], 'source': r['source'], 'split': splits['runs'][r['key']]} for r in runs]) \
    .pivot_table(index=['scenario', 'source'], columns='split', aggfunc='size', fill_value=0)

## 4. Lưu dữ liệu
`runs_raw.npz` (đầu vào cho feature selection và tune), `lstm_dataset.npz` (cửa sổ mặc định w = 12).

In [ ]:
fd.save_runs_npz(OUT / 'runs_raw.npz', runs, [], {'splits': splits, 'demo': DEMO,
                                                  'include_own_runs': INCLUDE_OWN_RUNS})
scalers = fd.fit_scalers(runs, splits, fd.NF)
windows = {s: fd.make_windows(runs, splits, s, DEFAULT_WINDOW, fd.HORIZON, fd.NF, scalers=scalers)
           for s in ('train', 'val', 'test')}
windows = {s: w for s, w in windows.items() if w is not None}
arrays = {}
for s, w in windows.items():
    arrays.update({f'{k}_{s}': (v.astype(str) if k == 'info' else v) for k, v in w.items()})
np.savez_compressed(OUT / 'lstm_dataset.npz', **arrays)
pd.DataFrame({s: {k: v.shape for k, v in w.items()} for s, w in windows.items()})

In [ ]:
import forecast_metrics as fm
w = windows.get('test') or windows.get('val')
if w is not None:
    d = {'Y_true': w['Y_raw'], 'Y_last': w['Y_last']}
    print('Baseline persistence (y_hat(t+k) = y(t)) trên', 'test' if 'test' in windows else 'val')
    display(fm.error_table(fm.persistence(d), w['Y_raw'], fd.TARGETS).round(4))